# 00_data_prep.ipynb — Agent 1: Data Extraction & Preprocessing

**Project:** SentinelFatal2 — Foundation Model for Fetal Distress Prediction from CTG  
**SSOT:** `docs/work_plan.md`  
**Output artifacts:** O1.1–O1.11 (see `docs/agent_workflow.md` → Agent 1)  
**Idempotent:** safe to re-run (exist_ok=True, already-processed files skipped)  

---

In [ ]:
# ─── Cell 1: Environment detection & path setup ───────────────────────────────
import os, sys

ON_COLAB = 'google.colab' in sys.modules

if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/SentinelFatal2'
else:
    # Local VS Code / Jupyter: root is one level up from notebooks/
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'ON_COLAB : {ON_COLAB}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'cwd: {os.getcwd()}')

In [ ]:
# ─── Cell 2: Pre-run gate checks ─────────────────────────────────────────────
import os

def gate_check(condition: bool, label: str, detail: str = '') -> None:
    status = '✅' if condition else '❌'
    msg = f'{status}  {label}'
    if detail:
        msg += f'  [{detail}]'
    print(msg)
    assert condition, f'Pre-run gate FAILED: {label}'

# G1.1 — archives exist
gate_check(
    os.path.exists('data/CTGDL/CTGDL_ctu_uhb_proc_csv.tar.gz'),
    'G1.1a  CTGDL_ctu_uhb_proc_csv.tar.gz exists'
)
gate_check(
    os.path.exists('data/CTGDL/CTGDL_FHRMA_proc_csv.tar.gz'),
    'G1.1b  CTGDL_FHRMA_proc_csv.tar.gz exists'
)

# G1.2 — metadata row count
import pandas as pd
meta = pd.read_csv('data/CTGDL/CTGDL_norm_metadata.csv')
gate_check(len(meta) == 981, 'G1.2  CTGDL_norm_metadata.csv has 981 rows', f'actual={len(meta)}')

# G1.3 — 552 .hea files
import glob
hea_files = glob.glob('data/ctu-chb-intrapartum-cardiotocography-database-1.0.0/ctu-chb-intrapartum-cardiotocography-database-1.0.0/*.hea')
gate_check(len(hea_files) == 552, 'G1.3  552 .hea files found', f'actual={len(hea_files)}')

print('\nAll pre-run gates passed.')

In [ ]:
# ─── Cell 3: Directory creation (idempotent) ─────────────────────────────────
import os

DIRS = [
    'data/raw/ctu_uhb',
    'data/raw/fhrma',
    'data/processed/ctu_uhb',
    'data/processed/fhrma',
    'data/splits',
    'src',
    'src/data',
    'notebooks',
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)
    print(f'  OK  {d}')

print('\nAll directories ready.')

In [ ]:
# ─── Cell 4: Archive extraction (idempotent — skip if already extracted) ─────
import subprocess, glob, os

def extract_archive(archive_path: str, target_dir: str, expected_count: int, strip: int = 1) -> None:
    existing = glob.glob(os.path.join(target_dir, '*.csv'))
    if len(existing) >= expected_count:
        print(f'  SKIP  {archive_path} — {len(existing)} CSVs already present')
        return
    print(f'  Extracting {archive_path} → {target_dir} ...')
    cmd = ['tar', '-xzf', archive_path, '-C', target_dir, f'--strip-components={strip}']
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'tar failed: {result.stderr}')
    count = len(glob.glob(os.path.join(target_dir, '*.csv')))
    assert count == expected_count, f'Expected {expected_count} CSVs, got {count} in {target_dir}'
    print(f'  OK  {count} CSVs extracted to {target_dir}')

extract_archive('data/CTGDL/CTGDL_ctu_uhb_proc_csv.tar.gz', 'data/raw/ctu_uhb', 552)
extract_archive('data/CTGDL/CTGDL_FHRMA_proc_csv.tar.gz',   'data/raw/fhrma',   135)

# Sanity: peek at column names
import pandas as pd
sample_ctg   = pd.read_csv(sorted(glob.glob('data/raw/ctu_uhb/*.csv'))[0])
sample_fhrma = pd.read_csv(sorted(glob.glob('data/raw/fhrma/*.csv'))[0])
print(f'\nCTU-UHB columns : {list(sample_ctg.columns)}')
print(f'FHRMA   columns : {list(sample_fhrma.columns)}')

In [ ]:
# ─── Cell 5: Preprocessing — FHR + UC → .npy (idempotent) ───────────────────
import glob, os, sys
import pandas as pd

sys.path.insert(0, os.getcwd())
from src.data.preprocessing import batch_process_dataset

meta = pd.read_csv('data/CTGDL/CTGDL_norm_metadata.csv')

def needs_processing(processed_dir: str, expected: int) -> bool:
    existing = glob.glob(os.path.join(processed_dir, '*.npy'))
    return len(existing) < expected

# CTU-UHB
ctg = meta[meta['dataset'] == 'ctg'].copy()
if needs_processing('data/processed/ctu_uhb', 552):
    print('Processing CTU-UHB...')
    batch_process_dataset(
        raw_dir='data/raw/ctu_uhb',
        processed_dir='data/processed/ctu_uhb',
        fname_col='fname', id_col='id',
        metadata_df=ctg, fhr_col='fhr', uc_col='uc', verbose=True
    )
else:
    print(f'SKIP CTU-UHB — {len(glob.glob("data/processed/ctu_uhb/*.npy"))} .npy already present')

# FHRMA
fhrma = meta[meta['dataset'] == 'fhrma'].copy()
if needs_processing('data/processed/fhrma', 135):
    print('Processing FHRMA...')
    batch_process_dataset(
        raw_dir='data/raw/fhrma',
        processed_dir='data/processed/fhrma',
        fname_col='fname', id_col='id',
        metadata_df=fhrma, fhr_col='fhr', uc_col='uc', verbose=True
    )
else:
    print(f'SKIP FHRMA — {len(glob.glob("data/processed/fhrma/*.npy"))} .npy already present')

print('\nPreprocessing complete.')

In [ ]:
# ─── Cell 6: Build ctu_uhb_clinical_full.csv from .hea files (idempotent) ────
import re, os, pandas as pd

OUTPUT_CSV = 'data/processed/ctu_uhb_clinical_full.csv'

if os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    if len(existing) == 552:
        print(f'SKIP — {OUTPUT_CSV} already has 552 rows')
    else:
        print(f'WARNING: {OUTPUT_CSV} has {len(existing)} rows — rebuilding...')
        os.remove(OUTPUT_CSV)

if not os.path.exists(OUTPUT_CSV):
    HEA_DIR = 'data/ctu-chb-intrapartum-cardiotocography-database-1.0.0/ctu-chb-intrapartum-cardiotocography-database-1.0.0/'
    FIELDS = {
        'pH':            r'#pH\s+([\d.]+)',
        'BDecf':         r'#BDecf\s+([\d.]+)',
        'Apgar1':        r'#Apgar1\s+(\d+)',
        'Apgar5':        r'#Apgar5\s+(\d+)',
        'gest_weeks':    r'#Gest\. weeks\s+(\d+)',
        'weight_g':      r'#Weight\(g\)\s+(\d+)',
        'presentation':  r'#Presentation\s+(\d+)',
        'induced':       r'#Induced\s+(\d+)',
        'stage1_min':    r'#I\.stage\s+(\d+)',
        'NoProgress':    r'#NoProgress\s+(\d+)',
        'stage2_min':    r'#II\.stage\s+(\d+)',
        'delivery_type': r'#Deliv\. type\s+(\d+)',
        'pos_stage2':    r'#Pos\. II\.st\.\s+(\d+)',
    }
    rows = []
    for fname in sorted(os.listdir(HEA_DIR)):
        if not fname.endswith('.hea'):
            continue
        record_id = fname.replace('.hea', '')
        content = open(os.path.join(HEA_DIR, fname), encoding='utf-8', errors='replace').read()
        row = {'record_id': record_id}
        for field, pattern in FIELDS.items():
            m = re.search(pattern, content)
            row[field] = m.group(1) if m else None
        rows.append(row)

    df = pd.DataFrame(rows)
    assert len(df) == 552, f'Expected 552, got {len(df)}'
    df.to_csv(OUTPUT_CSV, index=False)
    print(f'Created {OUTPUT_CSV} ({len(df)} rows)')

print('ctu_uhb_clinical_full.csv ready.')

In [ ]:
# ─── Cell 7: Create splits CSVs (idempotent) ─────────────────────────────────
import pandas as pd, os

meta = pd.read_csv('data/CTGDL/CTGDL_norm_metadata.csv')
ctg = meta[meta['dataset'] == 'ctg']

splits = {
    'train':   (ctg[ctg['test'] == 0][['id', 'target', 'fname']], 441),
    'val':     (ctg[ctg['test'] == 1][['id', 'target', 'fname']], 56),
    'test':    (ctg[ctg['test'] == 2][['id', 'target', 'fname']], 55),
}
pretrain_df = meta[meta['dataset'].isin(['ctg', 'fhrma'])][['id', 'dataset', 'fname']]

for name, (df, expected) in splits.items():
    path = f'data/splits/{name}.csv'
    df.to_csv(path, index=False)
    assert len(df) == expected, f'{name}: got {len(df)}, expected {expected}'
    print(f'  {path}: {len(df)} rows  ✅')

pretrain_df.to_csv('data/splits/pretrain.csv', index=False)
assert len(pretrain_df) == 687
print(f'  data/splits/pretrain.csv: {len(pretrain_df)} rows  ✅')

print('\nAll split CSVs created.')

In [ ]:

# ─── Cell 8: Validation V1.1 – V1.9 + G1.5 ──────────────────────────────────
import glob, os, random
import numpy as np
import pandas as pd

random.seed(42)
results = {}

def check(label: str, cond: bool, detail: str = '') -> None:
    status = '✅' if cond else '❌'
    msg = f'{status}  {label}'
    if detail:
        msg += f'  [{detail}]'
    print(msg)
    assert cond, f'FAILED: {label}  {detail}'
    results[label] = 'PASS'

# V1.1
npy_ctg = sorted(glob.glob('data/processed/ctu_uhb/*.npy'))
check('V1.1 552 ctg .npy exist', len(npy_ctg) == 552, f'actual={len(npy_ctg)}')
for f in random.sample(npy_ctg, 10):
    arr = np.load(f)
    check(f'V1.1 shape(2,T) {os.path.basename(f)}', arr.shape[0] == 2)
    check(f'V1.1 range[0,1] {os.path.basename(f)}', arr.min() >= 0.0 and arr.max() <= 1.0)

# V1.2
npy_fhrma = sorted(glob.glob('data/processed/fhrma/*.npy'))
check('V1.2 135 fhrma .npy exist', len(npy_fhrma) == 135, f'actual={len(npy_fhrma)}')
for f in random.sample(npy_fhrma, 10):
    arr = np.load(f)
    check(f'V1.2 shape(2,T) {os.path.basename(f)}', arr.shape[0] == 2)
    check(f'V1.2 range[0,1] {os.path.basename(f)}', arr.min() >= 0.0 and arr.max() <= 1.0)

# V1.3
sample_all = random.sample(npy_ctg, 5) + random.sample(npy_fhrma, 5)
for f in sample_all:
    arr = np.load(f)
    check(f'V1.3 FHR[0,1] {os.path.basename(f)}', arr[0].min() >= 0.0 and arr[0].max() <= 1.0)
    check(f'V1.3 UC [0,1] {os.path.basename(f)}', arr[1].min() >= 0.0 and arr[1].max() <= 1.0)

# V1.4
train = pd.read_csv('data/splits/train.csv')
val   = pd.read_csv('data/splits/val.csv')
test  = pd.read_csv('data/splits/test.csv')
check('V1.4 train=441, acidemia=90',  len(train)==441 and int(train['target'].sum())==90)
check('V1.4 val=56, acidemia=12',     len(val)==56   and int(val['target'].sum())==12)
check('V1.4 test=55, acidemia=11',    len(test)==55  and int(test['target'].sum())==11)

# V1.5
clin = pd.read_csv('data/processed/ctu_uhb_clinical_full.csv')
check('V1.5 552 rows in clinical_full', len(clin) == 552)
check('V1.5 NoProgress column exists', 'NoProgress' in clin.columns)

# V1.6 — Note: deviation S8: CTGDL uses pH <= 7.15 (inclusive), not pH < 7.15
meta    = pd.read_csv('data/CTGDL/CTGDL_norm_metadata.csv')
ctg_meta= meta[meta['dataset'] == 'ctg'].copy()
ctg_meta['id'] = ctg_meta['id'].astype(str).str.strip()
clin2   = clin.copy()
clin2['record_id'] = clin2['record_id'].astype(str).str.strip()
clin2['pH'] = pd.to_numeric(clin2['pH'], errors='coerce')
merged = ctg_meta.merge(clin2[['record_id', 'pH']], left_on='id', right_on='record_id')
check('V1.6 merge=552',   len(merged) == 552)
check('V1.6 pH notna',    merged['pH'].notna().all())
mis = merged[merged['target'] != (merged['pH'] <= 7.15).astype(int)]  # S8: <= not <
check('V1.6 0 pH-label mismatches (pH<=7.15, deviation S8)', len(mis) == 0, f'mismatches={len(mis)}')

# V1.7
check('V1.7 train∩val=∅',  len(set(train['id']) & set(val['id']))  == 0)
check('V1.7 train∩test=∅', len(set(train['id']) & set(test['id'])) == 0)
check('V1.7 val∩test=∅',   len(set(val['id'])   & set(test['id'])) == 0)

# V1.8
pretrain = pd.read_csv('data/splits/pretrain.csv')
check('V1.8 pretrain=687', len(pretrain) == 687)

# V1.9
check('V1.9 notebook exists', os.path.exists('notebooks/00_data_prep.ipynb'))

# G1.5 — no NaN in any .npy
nan_files = [f for f in (npy_ctg + npy_fhrma) if np.isnan(np.load(f)).any()]
check(f'G1.5 no NaN in any .npy (total={len(npy_ctg)+len(npy_fhrma)})', len(nan_files) == 0)

print(f'\n{"="*60}')
print(f'ALL VALIDATIONS PASSED  ({len(results)} checks)')
print(f'{"="*60}')
